# Week 7 scratch

Stuff we need to import (copy this to course site)

In [4]:
import csv
import json
import os # operating system library, let's us do things like making new folders
import requests
from time import sleep
from bs4 import BeautifulSoup # the scraping library!

Some example HTML (copy this to course site)

In [5]:
html = """
<html>
  <body>
    <h1>Adeline Ravoux</h1>
    <p class="artist">Vincent van Gogh</p>
    <p class="date">1890</p>
    <a href="/art/1958.31">Artwork page</a>
  </body>
</html>
"""

soup = BeautifulSoup(html, "html.parser")  # html.parser is the built-in parser that comes with python

Having a peek at what's in the soup using the find and find_all methods:

In [6]:
print(soup.find("h1"))
print(soup.find_all("p")) # finds all p elements
print(soup.find("p", class_="date")) # finds the first p element with the class  "date"

<h1>Adeline Ravoux</h1>
[<p class="artist">Vincent van Gogh</p>, <p class="date">1890</p>]
<p class="date">1890</p>


An element's attributes (such as "href", the link URL) are accessed like dictionary keys using square brackets:

In [7]:
link = soup.find("a") # access the first a element
print(link["href"])

/art/1958.31


## Setting up headers (identifying yourself) and fetching a page

(copy this into course site)

In [8]:
headers = {"User-Agent": "INFO664-class-project (educational)"}   # you can call this whatever you want, it shows up on the scraped website's logs as who you are (ie not a Google/AI bot or web browser)

os.makedirs("scraped_pages", exist_ok=True)  # make a new folder to hold our scraped website HTML data. exist_ok = True means, if the directory already exists, do nothing and don't return an error

def fetch_page(url, filename):  # declaring a new function, which takes a URL to be scraped, and a destination filename, as arguments
    if os.path.exists(filename):  # check to see if file already exists, if so, don't download it from the web, load it instead
        with open(filename, "r", encoding="utf-8") as file:
            return file.read()  # in this case, return provides the read file data, and escapes from the function, ignoring everything below

    response = requests.get(url, headers=headers) # grab the file, using our pre-defined headers as the header data

    if not response.ok: # if the response reports an error
        print(url, "failed with status code", response.status_code) # tell us why it failed
        return None # return a "None" object, since there's nothing usable in the response, and escape from this function
    with open(filename, "w",  encoding="utf-8") as file:
        file.write(response.text)

    sleep(10) # wait 10 seconds, since our robots.txt file asks for a 10 second Crawl-delay
    return response.text

In [53]:
url = "https://www.clevelandart.org/art/1958.31"  # URL copied from cleveland museum of art website

html = fetch_page(url, "scraped_pages/cma_1958_31.html")  # grabs the URL using our fetch function
soup = BeautifulSoup(html, "html.parser")  # soups it into a usable object

print(len(html)) # length of characters in HTML  

380630


Finding the title:

text works pretty much the same as [get_text()](https://www.crummy.com/software/BeautifulSoup/bs4/doc/#get-text), except get_text() is configurable 

In [54]:
print(soup.title.get_text())
print(soup.title.text)

Adeline Ravoux | Cleveland Museum of Art
Adeline Ravoux | Cleveland Museum of Art


Looking at all elements of a certain type and extracting an attribute (href)

In [55]:
all_links = soup.find_all("a")

for link in all_links:
    print(link.get_text(), link["href"])

Vincent van Gogh /art/collection/search?artists=Vincent%20van%20Gogh
Netherlands /art/collection/search?culture=Netherlands
oil on fabric /art/collection/search?medium=oil%20on%20fabric
Bequest of Leonard C. Hanna Jr. /art/collection/search?credit=Bequest%20of%20Leonard%20C.%20Hanna%20Jr.
CMA's Open Access Initiative. /open-access
222 Impressionism & Post-Impressionism /art/collection/search?gallery=222%20Impressionism%20%26%20Post-Impressionism
ArtLens App https://cma.org/artlens/artwork/1958.31
Print /print/art/1958.31
Collection /art/collection/search?collection_mapped=Modern%20Euro%20Painting
Department /art/collection/search?department=Modern%20European%20Painting%20and%20Sculpture
Type of artwork /art/collection/search?type=Painting
Medium /art/collection/search?medium_mapped=oil%20on%20canvas
Credit line /art/collection/search?credit=Bequest%20of%20Leonard%20C.%20Hanna%20Jr.
Two Poplars in the Alpilles Oval Sticker https://shop.clevelandart.org/products/two-poplars-in-the-alpill

Having looked at the structure of the HTML file in the browser, we have seen that all the details are stored in a container with an id called `artwork-details-container`. We can have a look at what's in there:

In [59]:
details = soup.find(id="artwork-details-container")
print(len(details))

children = details.find_all(True, recursive=False)  # True means find all elements, recursive means only find the top-level tags of the direct children and not any children thereof)

for index, child in enumerate(children):  # enumerate gives us both the contents of the list item, as well as its index number
    print(index, child.name, "|", child.get_text(" ", strip=True))  # print the index number, the type of element, and the text contents of each child element

14
0 h2 | Details / Information for Adeline Ravoux
1 div | Adeline Ravoux
2 div | 1890
3 div | Vincent van Gogh (Dutch, 1853–1890)
4 div | Culture Netherlands
5 div | Medium oil on fabric
6 div | Measurements Framed: 72.5 x 73.5 x 8.5 cm (28 9/16 x 28 15/16 x 3 3/8 in.); Unframed: 50.2 x 50.5 cm (19 3/4 x 19 7/8 in.)
7 div | Credit Line Bequest of Leonard C. Hanna Jr. 1958.31
8 div | Public Domain You can copy, modify, and distribute this work, all without asking permission. Learn more about CMA's Open Access Initiative.
9 div | Location 222 Impressionism & Post-Impressionism
10 div | ArtLens App Share Download Print
11 div | ? Did You Know? Adeline Ravoux, at age 13, was not pleased with her portrait and did not think the image resembled her. Today, a photograph exists of Adeline in her late seventies and the resemblance is truly remarkable.
12 div | Description In May 1890, Vincent van Gogh arrived in Auvers, a small town north of Paris, where he rented a room at the inn of Arthur Ra

Let's look at how to extract each element here, in a way that will replicate for any artwork

In [76]:
print(children[1])  # here i can see that this div has an "h3" child. the title will likely be the first h3 tag.
title = details.find("h3")
print(title.get_text())
print(children[3])  # i can see the artist's name is inside an "h4" child, so extract it the same way as above
artist = details.find("h4")
print(artist.get_text())

<div class="mb-5 md:mb-0 flex-col flex w-full max-w-[120rem] xl:mx-auto !max-w-full !mb-0" role="none"><h3 class="!mb-0 md:mb-4 xsm:mb-2 prose-h3" data-testid="heading" id="adeline-ravoux">Adeline Ravoux</h3><div class=""></div></div>
Adeline Ravoux
<div class="mt-2 md:mt-4 mb-4"><div class="flex items-baseline gap-3 flex-wrap justify-start mb-2"><div class="mb-5 md:mb-0 flex-col flex w-full max-w-[120rem] xl:mx-auto !w-auto xl:!mx-0 !mb-0" role="none"><h4 class="!mb-0 first-letter:!normal-case font-fold-grotesque md:mb-4 xsm:mb-2 prose-h4" data-testid="heading" id="-object-object-"><span><a class="!text-blue-03 font-bold" href="/art/collection/search?artists=Vincent%20van%20Gogh" target="_self"><span class="underline underline-offset-2">Vincent van Gogh</span></a></span></h4></div><span class="body-1-tight md:body-2-tight text-grey-05 whitespace-nowrap">(Dutch, 1853–1890)</span></div></div>
Vincent van Gogh


In [81]:
print(children[2])  # the date element doesn't have anything unique about it
#(no unique h3 tag or class name)
# so we will find it by finding the next div from the previous one

title_parent = title.parent
print(title_parent)
date = title_parent.find_next_sibling("div")  # find_next_sibling finds the next element that is also a "sibling"
print(date.get_text())

print(children[2].get_text()) # this might work also, but best to think about the order of things when "templating" using bs

<div class="mt-1 text-15 md:text-19 leading-[140%] md:leading-[180%] tracking-[0.01em] md:tracking-normal">1890</div>
<div class="mb-5 md:mb-0 flex-col flex w-full max-w-[120rem] xl:mx-auto !max-w-full !mb-0" role="none"><h3 class="!mb-0 md:mb-4 xsm:mb-2 prose-h3" data-testid="heading" id="adeline-ravoux">Adeline Ravoux</h3><div class=""></div></div>
1890
1890


In [99]:
artist_parent = artist.parent
print(artist_parent)
bio = artist_parent.find_next_sibling()
print(bio.get_text())

<div class="mb-5 md:mb-0 flex-col flex w-full max-w-[120rem] xl:mx-auto !w-auto xl:!mx-0 !mb-0" role="none"><h4 class="!mb-0 first-letter:!normal-case font-fold-grotesque md:mb-4 xsm:mb-2 prose-h4" data-testid="heading" id="-object-object-"><span><a class="!text-blue-03 font-bold" href="/art/collection/search?artists=Vincent%20van%20Gogh" target="_self"><span class="underline underline-offset-2">Vincent van Gogh</span></a></span></h4></div>
(Dutch, 1853–1890)


Finding the labeled data involves looking up the labels and then finding the next sibling

In [90]:
culture_label = details.find("div", string="Culture")
print(culture_label)
culture = culture_label.find_next_sibling()
print(culture.get_text())

<div class="font-bold mb-1 sm:mb-0">Culture</div>
Netherlands


In [89]:
medium_label = details.find("div", string="Medium")
print(medium_label)
medium = medium_label.find_next_sibling()
print(medium.get_text())

<div class="font-bold mb-1 sm:mb-0">Medium</div>
oil on fabric


In [100]:
artwork = {
    "Artist": artist.get_text(),
    "Title": title.get_text(),
    "Date": date.get_text(),
    "Medium": medium.get_text(),
    "Bio": bio.get_text()
}

print(artwork)

{'Artist': 'Vincent van Gogh', 'Title': 'Adeline Ravoux', 'Date': '1890', 'Medium': 'oil on fabric', 'Bio': '(Dutch, 1853–1890)'}


At this point, we can try the whole process again with a different URL and see if it correctly sorts the data